# 07: Gradient updates and parameter schedules

![Stable update pipeline](../images/07_gradient_updates_and_schedules.svg)

**Learning goals:** verify backpropagation, match full-batch gradients with accumulation, inspect Adam moments, configure AdamW parameter groups, clip gradients, schedule learning rates, and use CPU mixed precision safely.

In [ ]:
import math
import random
import numpy as np
import torch
from torch import nn

SEED = 7
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)
print(f'PyTorch {torch.__version__}, device=cpu')

## 1. Check backpropagation by hand

For `prediction = w*x` and `loss = 0.5*(prediction-y)^2`, the chain rule gives `dL/dw = (w*x-y)*x`. Autograd should produce the same scalar. Calling `backward()` accumulates into `.grad`.

In [ ]:
w = torch.tensor(2.0, requires_grad=True)
x, y = torch.tensor(3.0), torch.tensor(5.0)
loss = 0.5 * (w * x - y).square()
loss.backward()
manual = (w.detach() * x - y) * x
print(f'autograd={w.grad.item():.1f}, manual={manual.item():.1f}')
assert torch.allclose(w.grad, manual)

## 2. Gradient accumulation preserves a larger batch mean

For equal microbatches, dividing every mean loss by the number of microbatches makes accumulated gradients equal the full-batch mean gradient. `zero_grad(set_to_none=True)` avoids writing explicit zero tensors.

In [ ]:
X = torch.randn(16, 3)
target = torch.randn(16, 1)
model = nn.Linear(3, 1)
criterion = nn.MSELoss()

model.zero_grad(set_to_none=True)
criterion(model(X), target).backward()
full_grads = [p.grad.clone() for p in model.parameters()]

model.zero_grad(set_to_none=True)
K = 4
for xb, yb in zip(X.chunk(K), target.chunk(K)):
    (criterion(model(xb), yb) / K).backward()
micro_grads = [p.grad.clone() for p in model.parameters()]
max_difference = max((a - b).abs().max().item() for a, b in zip(full_grads, micro_grads))
print(f'max gradient difference={max_difference:.2e}')
assert max_difference < 1e-6

## 3. One scalar AdamW update

Adam tracks exponential averages `m` and `v`, corrects their zero-start bias, and normalizes the update coordinatewise. AdamW applies weight decay directly to the parameter instead of putting decay through those moments.

In [ ]:
theta, grad = 2.0, 0.5
beta1, beta2, lr, decay, eps = 0.9, 0.99, 0.1, 0.01, 1e-8
m = (1 - beta1) * grad
v = (1 - beta2) * grad**2
m_hat = m / (1 - beta1)
v_hat = v / (1 - beta2)
theta_new = theta * (1 - lr * decay) - lr * m_hat / (math.sqrt(v_hat) + eps)
print(f'm={m:.4f}, v={v:.4f}, updated theta={theta_new:.4f}')
assert abs(theta_new - 1.898) < 1e-6

## 4. A complete CPU training loop

We fit a small nonlinear classifier to synthetic points. Matrix weights receive decay, while one-dimensional bias and normalization parameters do not. We accumulate two microbatches, use bfloat16 autocast on CPU, clip the global gradient norm, then advance a linear-warmup plus cosine scheduler. Bfloat16 has a wide exponent range, so CPU training does not use float16 gradient scaling here.

In [ ]:
N = 512
features = torch.randn(N, 2)
labels = ((features[:, 0] * features[:, 1] + 0.2 * features[:, 0]) > 0).long()
net = nn.Sequential(nn.Linear(2, 32), nn.LayerNorm(32), nn.GELU(), nn.Linear(32, 2))
decay_params, no_decay_params = [], []
for name, parameter in net.named_parameters():
    (no_decay_params if parameter.ndim == 1 or name.endswith('bias') else decay_params).append(parameter)
all_ids = [id(p) for p in decay_params + no_decay_params]
assert len(all_ids) == len(set(all_ids)) == len(list(net.parameters()))
optimizer = torch.optim.AdamW([
    {'params': decay_params, 'weight_decay': 0.03},
    {'params': no_decay_params, 'weight_decay': 0.0},
], lr=3e-3)
print(f'decayed tensors={len(decay_params)}, non-decayed tensors={len(no_decay_params)}')

In [ ]:
TOTAL_STEPS, WARMUP = 60, 8
def lr_multiplier(step):
    if step < WARMUP:
        return (step + 1) / WARMUP
    progress = min(1.0, (step - WARMUP) / max(1, TOTAL_STEPS - WARMUP - 1))
    return 0.05 + 0.95 * 0.5 * (1 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_multiplier)
generator = torch.Generator().manual_seed(SEED)
learning_rates, grad_norms = [], []
for step in range(TOTAL_STEPS):
    indices = torch.randint(0, N, (64,), generator=generator)
    optimizer.zero_grad(set_to_none=True)
    for idx in indices.chunk(2):
        with torch.autocast(device_type='cpu', dtype=torch.bfloat16):
            micro_loss = nn.functional.cross_entropy(net(features[idx]), labels[idx]) / 2
        micro_loss.backward()
    norm = torch.nn.utils.clip_grad_norm_(net.parameters(), max_norm=1.0)
    optimizer.step()
    learning_rates.append(optimizer.param_groups[0]['lr'])
    grad_norms.append(float(norm))
    scheduler.step()

with torch.no_grad():
    accuracy = (net(features).argmax(dim=1) == labels).float().mean().item()
print(f'accuracy={accuracy:.3f}, peak lr={max(learning_rates):.4g}, final lr={learning_rates[-1]:.4g}')
print(f'max pre-clip gradient norm={max(grad_norms):.3f}')
assert accuracy > 0.80
assert all(math.isfinite(x) for x in grad_norms)

## 5. Fix exposure and validate resumes

With effective batch size $B$ and $U$ completed updates, sampled-example exposure is $C=BU$. Every condition in a comparison should use the same planned exposure and final checkpoint. An outcome-blind eight-job concurrent speed probe may choose one common tier before outcomes are viewed. Exact resume requires the same frozen metadata fields and values, including every named random stream.

In [ ]:
EFFECTIVE_BATCH = 64
planned_exposure = TOTAL_STEPS * EFFECTIVE_BATCH
assert planned_exposure == 3_840

def exposure_tier(concurrent_rates, *, storage_stable):
    rates = np.asarray(concurrent_rates, dtype=float)
    if rates.shape != (8,) or not np.isfinite(rates).all() or np.any(rates <= 0):
        raise ValueError('require eight positive finite concurrent per-GPU rates')
    if not storage_stable:
        return None
    slowest_rate = rates.min()
    if slowest_rate >= 60:
        return 8_192_000
    if slowest_rate >= 30:
        return 4_096_000
    return None

assert exposure_tier([75] * 8, storage_stable=True) == 8_192_000
assert exposure_tier([65] * 7 + [45], storage_stable=True) == 4_096_000
assert exposure_tier([65] * 7 + [20], storage_stable=True) is None
assert exposure_tier([75] * 8, storage_stable=False) is None
REQUIRED_RESUME_FIELDS = {
    'manifest_digest', 'sequence_support', 'window_policy', 'planned_exposure',
    'effective_batch', 'completed_updates', 'optimization_seed',
    'sequence_stream_version', 'temporal_stream_version',
    'spatial_stream_version', 'mask_stream_version',
}
CANONICAL_SUPPORTS = {'low', 'high'}
CANONICAL_WINDOW_POLICIES = {'frozen_random', 'resampled_anchor'}
saved_provenance = {
    'manifest_digest': 'synthetic-manifest-v1', 'sequence_support': 'low',
    'window_policy': 'resampled_anchor',
    'planned_exposure': planned_exposure, 'effective_batch': EFFECTIVE_BATCH,
    'completed_updates': TOTAL_STEPS, 'optimization_seed': SEED,
    'sequence_stream_version': 'sequence-v1',
    'temporal_stream_version': 'temporal-v1',
    'spatial_stream_version': 'spatial-v1',
    'mask_stream_version': 'mask-v1',
}
def validate_resume(saved, requested):
    saved_fields, requested_fields = set(saved), set(requested)
    if saved_fields != REQUIRED_RESUME_FIELDS or requested_fields != REQUIRED_RESUME_FIELDS:
        saved_missing = REQUIRED_RESUME_FIELDS - saved_fields
        saved_extra = saved_fields - REQUIRED_RESUME_FIELDS
        requested_missing = REQUIRED_RESUME_FIELDS - requested_fields
        requested_extra = requested_fields - REQUIRED_RESUME_FIELDS
        raise ValueError(
            f'resume fields differ: saved_missing={sorted(saved_missing)}, '
            f'saved_extra={sorted(saved_extra)}, requested_missing={sorted(requested_missing)}, '
            f'requested_extra={sorted(requested_extra)}')
    for label, record in (('saved', saved), ('requested', requested)):
        if record['sequence_support'] not in CANONICAL_SUPPORTS:
            raise ValueError(f'{label} sequence_support is not canonical')
        if record['window_policy'] not in CANONICAL_WINDOW_POLICIES:
            raise ValueError(f'{label} window_policy is not canonical')
    mismatches = [key for key in sorted(REQUIRED_RESUME_FIELDS)
                  if requested[key] != saved[key]]
    if mismatches:
        raise ValueError(f'resume provenance mismatch: {mismatches}')

requested_provenance = saved_provenance.copy()
validate_resume(saved_provenance, requested_provenance)
changed = requested_provenance | {'manifest_digest': 'different-manifest'}
try:
    validate_resume(saved_provenance, changed)
except ValueError as error:
    assert 'manifest_digest' in str(error)
else:
    raise AssertionError('a changed manifest must stop resume')
extra_field = requested_provenance | {'new_stream_version': 'unexpected-v1'}
try:
    validate_resume(saved_provenance, extra_field)
except ValueError as error:
    assert 'requested_extra' in str(error)
else:
    raise AssertionError('an extra resume field must stop resume')
noncanonical = requested_provenance | {'sequence_support': 'L', 'window_policy': 'R'}
try:
    validate_resume(noncanonical, noncanonical)
except ValueError as error:
    assert 'canonical' in str(error)
else:
    raise AssertionError('matching noncanonical labels must stop resume')
print(f'exposure={planned_exposure}, final update={TOTAL_STEPS}, resume identity verified')

## Efficiency, exercises, and takeaways

`autocast` chooses operation precision without permanently converting model parameters. `clip_grad_norm_` computes and applies one global scale. `LambdaLR` changes each parameter group's learning rate without rebuilding the optimizer. Avoid frequent `.item()` calls in accelerator training because they can force synchronization.

1. Remove `/ 2` from the microbatch loss. **Check:** gradient norms become about twice as large before clipping.
2. Set `max_norm=0.01`. **Check:** clipping happens almost every step, which changes optimization rather than merely guarding rare spikes.
3. Save `optimizer.state_dict()` after training. **Check:** it contains moment tensors and a step count in addition to hyperparameters.

**Takeaways:** backward computes and accumulates gradients; AdamW transforms them using moment history and separate decay; parameter groups express intentional exceptions; and clipping, scheduling, and precision must be applied in the correct order.

## Continue learning

[Previous notebook: 06](06_representation_collapse.ipynb) | [Lecture](../lectures/07_gradient_updates_and_schedules.md) | [Curriculum](../README.md) | [Next notebook: 08](08_group_aware_sampling.ipynb)